# BugBuster HAT — RP2040 Firmware Build & Flash

**Architecture:** CMake + Ninja · arm-none-eabi-gcc (ARM-provided, not Homebrew 15.x) · picotool flash · BBP v4 HAT commands via `BugBuster` Python library  
**Last refreshed:** 2026-05-04

Builds the RP2040 HAT firmware from source and (optionally) flashes it,
then demonstrates HAT commands via the Python library.

| Section | Action |
|---|---|
| 1 | Dependency check |
| 2 | CMake configure + Ninja build |
| 3 | Flash via picotool (needs board in BOOTSEL) |
| 4 | HAT demo via Python library |

> **macOS note:** Homebrew arm-gcc 15.x has no sysroot — use the ARM-provided
> binary from [developer.arm.com](https://developer.arm.com/downloads/-/arm-gnu-toolchain-downloads).
>
> **LA-done IRQ** and **1 MHz/4-ch streaming** are firmware-implemented;
> bench validation pending.

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

REPO_ROOT  = Path(os.getcwd()).parent
FW_DIR     = REPO_ROOT / 'Firmware' / 'RP2040'
BUILD_DIR  = FW_DIR / 'build'
UF2_PATH   = BUILD_DIR / 'bugbuster_hat.uf2'

print(f'Repo root  : {REPO_ROOT}')
print(f'Firmware   : {FW_DIR}')
print(f'Build dir  : {BUILD_DIR}')
print(f'FW exists  : {FW_DIR.exists()}')

## 1. Dependency check

In [ ]:
def _ver(cmd):
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        out = (r.stdout + r.stderr).strip()
        return out.splitlines()[0] if out else '?'
    except Exception:
        return None

checks = [
    ('cmake',              ['cmake', '--version']),
    ('ninja',              ['ninja', '--version']),
    ('arm-none-eabi-gcc',  ['arm-none-eabi-gcc', '--version']),
    ('arm-none-eabi-g++',  ['arm-none-eabi-g++', '--version']),
    ('picotool',           ['picotool', '--version']),
    ('git',                ['git',  '--version']),
]

missing = []
for name, cmd in checks:
    ver = _ver(cmd)
    if ver:
        print(f'  OK  {name}: {ver[:60]}')
    else:
        print(f'  --  {name}: NOT FOUND')
        if name != 'picotool':   # picotool is optional (can use UF2 drag-drop)
            missing.append(name)

if missing:
    print(f'\nMissing required tools: {missing}')
    print('Install arm-none-eabi-gcc from https://developer.arm.com/downloads/-/arm-gnu-toolchain-downloads')
else:
    print('\nAll required tools found.')

## 2. CMake configure + Ninja build

Configures out-of-tree in `Firmware/RP2040/build/` and builds with Ninja.

In [ ]:
BUILD_DIR.mkdir(parents=True, exist_ok=True)

# Configure
cmake_cmd = [
    'cmake', '-G', 'Ninja',
    '-DCMAKE_BUILD_TYPE=RelWithDebInfo',
    '..'
]
print('--- cmake configure ---')
r = subprocess.run(cmake_cmd, cwd=str(BUILD_DIR))
print(f'cmake exit: {r.returncode}')

In [ ]:
import multiprocessing
nproc = multiprocessing.cpu_count()

print(f'--- ninja build ({nproc} jobs) ---')
r = subprocess.run(['ninja', f'-j{nproc}'], cwd=str(BUILD_DIR))
print(f'ninja exit: {r.returncode}')

if UF2_PATH.exists():
    size_kb = UF2_PATH.stat().st_size / 1024
    print(f'UF2: {UF2_PATH}  ({size_kb:.0f} KB)')
else:
    print(f'UF2 not found at {UF2_PATH} — check build output above.')

## 3. Flash via picotool

> **User must run** — requires RP2040 in BOOTSEL mode.
>
> To enter BOOTSEL: hold the BOOTSEL button while connecting USB (or while pressing reset).
> The board appears as a USB mass-storage device called `RPI-RP2`.

The flash cell is commented out to prevent accidental execution.

In [ ]:
# Run in a terminal (or uncomment here when board is in BOOTSEL mode):
# !picotool load -x "{UF2_PATH}"
print(f'Flash command: picotool load -x {UF2_PATH}')
print('Uncomment the line above and run when board is in BOOTSEL mode.')

## 4. HAT demo via Python library

After the ESP32 attaches the HAT, use `BugBuster` over USB to send HAT commands.
Requires both boards connected (ESP32-S3 with USB, RP2040 HAT attached via HAT connector).

In [ ]:
import sys, os
sys.path.insert(0, str(REPO_ROOT / 'python'))
import bugbuster

USB_PORT = os.environ.get('BUGBUSTER_PORT', None)

if USB_PORT is None:
    try:
        import serial.tools.list_ports
        ports = [p.device for p in serial.tools.list_ports.comports()
                 if p.vid == 0x303A]
        USB_PORT = ports[0] if ports else None
    except ImportError:
        pass

print(f'USB_PORT = {USB_PORT!r}')

In [ ]:
bb = None

try:
    if USB_PORT is None:
        raise RuntimeError('No USB port configured.')

    bb = bugbuster.connect_usb(USB_PORT)

    # Detect HAT
    hat = bb.hat_detect()
    print('--- hat_detect ---')
    for k, v in hat.items():
        print(f'  {k}: {v}')

    if not hat.get('present', False):
        print('\nHAT not detected. Attach the RP2040 HAT and retry.')
    else:
        # HAT status
        status = bb.hat_get_status()
        print('\n--- hat_get_status ---')
        for k, v in status.items():
            print(f'  {k}: {v}')

        # Power connector 0 on
        ok = bb.hat_set_power(0, True)
        print(f'\nhat_set_power(0, True) -> {ok}')

        # IO voltage 3.3V
        ok = bb.hat_set_io_voltage(3300)
        print(f'hat_set_io_voltage(3300 mV) -> {ok}')

        # LA arm + status
        la_arm = bb.hat_la_arm()
        print(f'hat_la_arm() -> {la_arm}')

        la_status = bb.hat_la_get_status()
        print('hat_la_get_status():')
        for k, v in la_status.items():
            print(f'  {k}: {v}')

except Exception as e:
    print(f'[No board / HAT] {e}')
finally:
    if bb is not None:
        try:
            bb.disconnect()
        except Exception:
            pass